In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [3]:
df = pd.read_csv("cleaned_fx_data.csv")

# Rename the first column as 'date'
df.rename(columns={df.columns[0]: 'date'}, inplace=True)

# Convert to datetime
df['date'] = pd.to_datetime(df['date'])

df.head()



,date,EUR,GBP,JPY,USD,USD_Return,USD_Volatility,GBP_Return,GBP_Volatility,EUR_Return,EUR_Volatility,JPY_Return,JPY_Volatility
0,2016-01-01,71.8627,97.6059,55.01,66.1780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-01-02,71.8627,97.6059,55.01,66.1780,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN
2,2016-01-03,71.8627,97.6059,55.01,66.1780,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN
3,2016-01-04,72.3907,97.9455,55.65,66.4623,0.004296,NaN,0.003479,NaN,0.007347,NaN,0.011634,NaN
4,2016-01-05,72.0315,97.9562,55.71,66.5418,0.001196,NaN,0.000109,NaN,-0.004962,NaN,0.001078,NaN


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3654 entries, 0 to 3653
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            3654 non-null   datetime64[ns]
 1   EUR             3654 non-null   float64       
 2   GBP             3654 non-null   float64       
 3   JPY             3654 non-null   float64       
 4   USD             3654 non-null   float64       
 5   USD_Return      3653 non-null   float64       
 6   USD_Volatility  3624 non-null   float64       
 7   GBP_Return      3653 non-null   float64       
 8   GBP_Volatility  3624 non-null   float64       
 9   EUR_Return      3653 non-null   float64       
 10  EUR_Volatility  3624 non-null   float64       
 11  JPY_Return      3653 non-null   float64       
 12  JPY_Volatility  3624 non-null   float64       
dtypes: datetime64[ns](1), float64(12)
memory usage: 371.2 KB


In [5]:
CURRENCIES = ['USD', 'EUR', 'GBP', 'JPY']
FORECAST_DAYS = [7, 30]


In [6]:
def backtest_prophet(data, currency):
    temp = data[['date', currency]].rename(
        columns={'date': 'ds', currency: 'y'}
    )
    
    train_data = temp.iloc[:-14]
    test_data = temp.iloc[-14:]
    
    model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True
    )
    
    model.fit(train_data)
    
    future = model.make_future_dataframe(periods=14)
    forecast = model.predict(future).tail(14)
    
    mae = mean_absolute_error(test_data['y'], forecast['yhat'])
    rmse = np.sqrt(mean_squared_error(test_data['y'], forecast['yhat']))
    
    return mae, rmse


In [7]:
currency = "USD"
usd_df = df[['date', currency]].rename(
    columns={'date': 'ds', currency: 'y'}
)

usd_df.tail()


,ds,y
3649,2025-12-28,89.8296
3650,2025-12-29,89.9756
3651,2025-12-30,89.9429
3652,2025-12-31,89.9198
3653,2026-01-01,89.9792


In [8]:
usd_train = usd_df.iloc[:-14]
usd_test = usd_df.iloc[-14:]

print("Train size:", len(usd_train))
print("Test size:", len(usd_test))
usd_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

usd_model.fit(usd_train)
usd_future = usd_model.make_future_dataframe(periods=14)
usd_forecast = usd_model.predict(usd_future)

usd_forecast_14 = usd_forecast.tail(14)
usd_forecast_14[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
usd_mae = mean_absolute_error(usd_test['y'], usd_forecast_14['yhat'])
usd_rmse = np.sqrt(mean_squared_error(usd_test['y'], usd_forecast_14['yhat']))

print(f"USD MAE  : {usd_mae:.4f}")
print(f"USD RMSE : {usd_rmse:.4f}")
backtest_results = {
    "USD": {"MAE": round(usd_mae, 4), "RMSE": round(usd_rmse, 4)}
}

backtest_results


Train size: 3640
Test size: 14


22:05:39 - cmdstanpy - INFO - Chain [1] start processing
22:05:43 - cmdstanpy - INFO - Chain [1] done processing


USD MAE  : 1.3162
USD RMSE : 1.3295


{'USD': {'MAE': 1.3162, 'RMSE': np.float64(1.3295)}}

In [10]:
usd_full_df = df[['date', 'USD']].rename(
    columns={'date': 'ds', 'USD': 'y'}
)

usd_full_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

usd_full_model.fit(usd_full_df)
usd_future_7 = usd_full_model.make_future_dataframe(periods=7)
usd_forecast_7 = usd_full_model.predict(usd_future_7)

usd_last_7 = usd_forecast_7.tail(1).iloc[0]

print("USD – 7 Day Forecast\n")
print("Neutral (Expected):", round(usd_last_7['yhat'], 2))
print("Optimistic (INR strengthens):", round(usd_last_7['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(usd_last_7['yhat_upper'], 2))



22:12:48 - cmdstanpy - INFO - Chain [1] start processing
22:12:49 - cmdstanpy - INFO - Chain [1] done processing


USD – 7 Day Forecast

Neutral (Expected): 88.59
Optimistic (INR strengthens): 87.26
Pessimistic (INR weakens): 89.9


In [11]:
usd_future_30 = usd_full_model.make_future_dataframe(periods=30)
usd_forecast_30 = usd_full_model.predict(usd_future_30)

usd_last_30 = usd_forecast_30.tail(1).iloc[0]

print("USD – 30 Day Forecast\n")
print("Neutral (Expected):", round(usd_last_30['yhat'], 2))
print("Optimistic (INR strengthens):", round(usd_last_30['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(usd_last_30['yhat_upper'], 2))


USD – 30 Day Forecast

Neutral (Expected): 89.07
Optimistic (INR strengthens): 87.67
Pessimistic (INR weakens): 90.44


In [12]:
currency = "EUR"
eur_df = df[['date', currency]].rename(
    columns={'date': 'ds', currency: 'y'}
)

eur_df.tail()
eur_train = eur_df.iloc[:-14]
eur_test = eur_df.iloc[-14:]

print("Train size:", len(eur_train))
print("Test size:", len(eur_test))

eur_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

eur_model.fit(eur_train)
eur_future = eur_model.make_future_dataframe(periods=14)
eur_forecast = eur_model.predict(eur_future)

eur_forecast_14 = eur_forecast.tail(14)
eur_forecast_14[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

eur_mae = mean_absolute_error(eur_test['y'], eur_forecast_14['yhat'])
eur_rmse = np.sqrt(mean_squared_error(eur_test['y'], eur_forecast_14['yhat']))

print(f"EUR MAE  : {eur_mae:.4f}")
print(f"EUR RMSE : {eur_rmse:.4f}")

backtest_results = {
    "EUR": {"MAE": round(eur_mae, 4), "RMSE": round(eur_rmse, 4)}
}

backtest_results

Train size: 3640
Test size: 14


22:24:03 - cmdstanpy - INFO - Chain [1] start processing
22:24:05 - cmdstanpy - INFO - Chain [1] done processing


EUR MAE  : 4.5572
EUR RMSE : 4.5671


{'EUR': {'MAE': 4.5572, 'RMSE': np.float64(4.5671)}}

In [13]:
# Full model for forecasting
eur_full_df = df[['date', 'EUR']].rename(
    columns={'date': 'ds', 'EUR': 'y'}
)

eur_full_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

eur_full_model.fit(eur_full_df)

# 7-day forecast
eur_future_7 = eur_full_model.make_future_dataframe(periods=7)
eur_forecast_7 = eur_full_model.predict(eur_future_7)

eur_last_7 = eur_forecast_7.tail(1).iloc[0]

print("EUR – 7 Day Forecast\n")
print("Neutral (Expected):", round(eur_last_7['yhat'], 2))
print("Optimistic (INR strengthens):", round(eur_last_7['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(eur_last_7['yhat_upper'], 2))

22:24:26 - cmdstanpy - INFO - Chain [1] start processing
22:24:28 - cmdstanpy - INFO - Chain [1] done processing


EUR – 7 Day Forecast

Neutral (Expected): 101.53
Optimistic (INR strengthens): 98.28
Pessimistic (INR weakens): 104.46


In [14]:
# 30-day forecast
eur_future_30 = eur_full_model.make_future_dataframe(periods=30)
eur_forecast_30 = eur_full_model.predict(eur_future_30)

eur_last_30 = eur_forecast_30.tail(1).iloc[0]

print("EUR – 30 Day Forecast\n")
print("Neutral (Expected):", round(eur_last_30['yhat'], 2))
print("Optimistic (INR strengthens):", round(eur_last_30['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(eur_last_30['yhat_upper'], 2))

EUR – 30 Day Forecast

Neutral (Expected): 102.37
Optimistic (INR strengthens): 99.32
Pessimistic (INR weakens): 105.17


In [15]:
currency = "GBP"
gbp_df = df[['date', currency]].rename(
    columns={'date': 'ds', currency: 'y'}
)

gbp_df.tail()
gbp_train = gbp_df.iloc[:-14]
gbp_test = gbp_df.iloc[-14:]

print("Train size:", len(gbp_train))
print("Test size:", len(gbp_test))

gbp_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

gbp_model.fit(gbp_train)
gbp_future = gbp_model.make_future_dataframe(periods=14)
gbp_forecast = gbp_model.predict(gbp_future)

gbp_forecast_14 = gbp_forecast.tail(14)
gbp_forecast_14[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

gbp_mae = mean_absolute_error(gbp_test['y'], gbp_forecast_14['yhat'])
gbp_rmse = np.sqrt(mean_squared_error(gbp_test['y'], gbp_forecast_14['yhat']))

print(f"GBP MAE  : {gbp_mae:.4f}")
print(f"GBP RMSE : {gbp_rmse:.4f}")

backtest_results = {
    "GBP": {"MAE": round(gbp_mae, 4), "RMSE": round(gbp_rmse, 4)}
}

backtest_results

Train size: 3640
Test size: 14


22:25:44 - cmdstanpy - INFO - Chain [1] start processing
22:25:45 - cmdstanpy - INFO - Chain [1] done processing


GBP MAE  : 2.7959
GBP RMSE : 2.8475


{'GBP': {'MAE': 2.7959, 'RMSE': np.float64(2.8475)}}

In [16]:
# Full model for forecasting
gbp_full_df = df[['date', 'GBP']].rename(
    columns={'date': 'ds', 'GBP': 'y'}
)

gbp_full_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

gbp_full_model.fit(gbp_full_df)

# 7-day forecast
gbp_future_7 = gbp_full_model.make_future_dataframe(periods=7)
gbp_forecast_7 = gbp_full_model.predict(gbp_future_7)

gbp_last_7 = gbp_forecast_7.tail(1).iloc[0]

print("GBP – 7 Day Forecast\n")
print("Neutral (Expected):", round(gbp_last_7['yhat'], 2))
print("Optimistic (INR strengthens):", round(gbp_last_7['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(gbp_last_7['yhat_upper'], 2))

22:26:00 - cmdstanpy - INFO - Chain [1] start processing
22:26:01 - cmdstanpy - INFO - Chain [1] done processing


GBP – 7 Day Forecast

Neutral (Expected): 118.64
Optimistic (INR strengthens): 115.37
Pessimistic (INR weakens): 121.84


In [17]:
# 30-day forecast
gbp_future_30 = gbp_full_model.make_future_dataframe(periods=30)
gbp_forecast_30 = gbp_full_model.predict(gbp_future_30)

gbp_last_30 = gbp_forecast_30.tail(1).iloc[0]

print("GBP – 30 Day Forecast\n")
print("Neutral (Expected):", round(gbp_last_30['yhat'], 2))
print("Optimistic (INR strengthens):", round(gbp_last_30['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(gbp_last_30['yhat_upper'], 2))

GBP – 30 Day Forecast

Neutral (Expected): 120.16
Optimistic (INR strengthens): 116.79
Pessimistic (INR weakens): 123.35


In [18]:
currency = "JPY"
jpy_df = df[['date', currency]].rename(
    columns={'date': 'ds', currency: 'y'}
)

jpy_df.tail()
jpy_train = jpy_df.iloc[:-14]
jpy_test = jpy_df.iloc[-14:]

print("Train size:", len(jpy_train))
print("Test size:", len(jpy_test))

jpy_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

jpy_model.fit(jpy_train)
jpy_future = jpy_model.make_future_dataframe(periods=14)
jpy_forecast = jpy_model.predict(jpy_future)

jpy_forecast_14 = jpy_forecast.tail(14)
jpy_forecast_14[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

jpy_mae = mean_absolute_error(jpy_test['y'], jpy_forecast_14['yhat'])
jpy_rmse = np.sqrt(mean_squared_error(jpy_test['y'], jpy_forecast_14['yhat']))

print(f"JPY MAE  : {jpy_mae:.4f}")
print(f"JPY RMSE : {jpy_rmse:.4f}")

backtest_results = {
    "JPY": {"MAE": round(jpy_mae, 4), "RMSE": round(jpy_rmse, 4)}
}

backtest_results

Train size: 3640
Test size: 14


22:27:16 - cmdstanpy - INFO - Chain [1] start processing
22:27:17 - cmdstanpy - INFO - Chain [1] done processing


JPY MAE  : 1.3429
JPY RMSE : 1.3590


{'JPY': {'MAE': 1.3429, 'RMSE': np.float64(1.359)}}

In [19]:
# Full model for forecasting
jpy_full_df = df[['date', 'JPY']].rename(
    columns={'date': 'ds', 'JPY': 'y'}
)

jpy_full_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

jpy_full_model.fit(jpy_full_df)

# 7-day forecast
jpy_future_7 = jpy_full_model.make_future_dataframe(periods=7)
jpy_forecast_7 = jpy_full_model.predict(jpy_future_7)

jpy_last_7 = jpy_forecast_7.tail(1).iloc[0]

print("JPY – 7 Day Forecast\n")
print("Neutral (Expected):", round(jpy_last_7['yhat'], 2))
print("Optimistic (INR strengthens):", round(jpy_last_7['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(jpy_last_7['yhat_upper'], 2))

22:27:48 - cmdstanpy - INFO - Chain [1] start processing
22:27:49 - cmdstanpy - INFO - Chain [1] done processing


JPY – 7 Day Forecast

Neutral (Expected): 58.73
Optimistic (INR strengthens): 56.43
Pessimistic (INR weakens): 61.03


In [20]:
# 30-day forecast
jpy_future_30 = jpy_full_model.make_future_dataframe(periods=30)
jpy_forecast_30 = jpy_full_model.predict(jpy_future_30)

jpy_last_30 = jpy_forecast_30.tail(1).iloc[0]

print("JPY – 30 Day Forecast\n")
print("Neutral (Expected):", round(jpy_last_30['yhat'], 2))
print("Optimistic (INR strengthens):", round(jpy_last_30['yhat_lower'], 2))
print("Pessimistic (INR weakens):", round(jpy_last_30['yhat_upper'], 2))

JPY – 30 Day Forecast

Neutral (Expected): 59.33
Optimistic (INR strengthens): 56.84
Pessimistic (INR weakens): 61.7
